# 5-bob. Xavfsizlikda sun'iy intellekt modelini baholash
## Kengaytirilgan amaliy ish: bootstrap yordamida ishonch oraliqlari

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/05-bob/5-1_bootstrap-yordamida-ishonch-oraliqlari.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 5-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Quyidagi dastur ikki modelning qarorlarini modellashtiradi. Haqiqatda A model hujumlarning taxminan 80 foizini, B model esa 87 foizini aniqlaydi, B esa biroz ko'proq noto'g'ri signal beradi. Modellarning xatolari bog'liq: ikkalasi ham eng "qiyin" hujumlarni o'tkazib yuboradi. Tajriba ikki xil sinov to'plamida — 30 ta va 300 ta hujumli to'plamlarda o'tkaziladi, har birida 10 000 ta oddiy yozuv bor.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np

rng = np.random.default_rng(5)

def sinov_toplami(n_hujum, n_oddiy=10_000):
    """Ikki modelning bir xil sinov to'plamidagi qarorlari (1 - "hujum" deb topdi)."""
    y = np.r_[np.ones(n_hujum, int), np.zeros(n_oddiy, int)]
    qiyinlik = rng.random(n_hujum)              # hujumning "qiyinligi" ikkala model uchun umumiy
    A = np.r_[qiyinlik < 0.80, rng.random(n_oddiy) < 0.004].astype(int)   # A: to'liqlik ~0,80
    B = np.r_[qiyinlik < 0.87, rng.random(n_oddiy) < 0.006].astype(int)   # B: to'liqlik ~0,87
    return y, A, B

def toliqlik(y, p):  return p[y == 1].mean()
def aniqlik(y, p):   return (p & y).sum() / max(p.sum(), 1)

def bootstrap(y, A, B, n=2000):
    """Sinov to'plamidan qaytarib tanlash: har safar bir xil yozuvlar ikkala model uchun olinadi."""
    natija = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        natija.append((toliqlik(y[i], A[i]), toliqlik(y[i], B[i]), aniqlik(y[i], A[i]), aniqlik(y[i], B[i])))
    return np.array(natija)

for n_hujum in (30, 300):
    y, A, B = sinov_toplami(n_hujum)
    r = bootstrap(y, A, B)
    oraliq = lambda v: f"[{np.percentile(v, 2.5):.2f}; {np.percentile(v, 97.5):.2f}]"
    farq = r[:, 1] - r[:, 0]
    print(f"Sinov to'plamida {n_hujum} ta hujum:")
    print(f"  to'liqlik  A = {toliqlik(y, A):.2f} {oraliq(r[:, 0])}   B = {toliqlik(y, B):.2f} {oraliq(r[:, 1])}")
    print(f"  aniqlik    A = {aniqlik(y, A):.2f} {oraliq(r[:, 2])}   B = {aniqlik(y, B):.2f} {oraliq(r[:, 3])}")
    print(f"  to'liqlik farqi B - A = {farq.mean():+.2f} {oraliq(farq)},  farq <= 0 bo'lgan holatlar: {(farq <= 0).mean():.1%}\n")

Sinov to'plamida 30 ta hujum:
  to'liqlik  A = 0.67 [0.48; 0.83]   B = 0.77 [0.59; 0.91]
  aniqlik    A = 0.34 [0.22; 0.47]   B = 0.31 [0.21; 0.42]
  to'liqlik farqi B - A = +0.10 [0.00; 0.21],  farq <= 0 bo'lgan holatlar: 4.9%



Sinov to'plamida 300 ta hujum:
  to'liqlik  A = 0.82 [0.77; 0.86]   B = 0.89 [0.85; 0.92]
  aniqlik    A = 0.85 [0.81; 0.89]   B = 0.79 [0.75; 0.84]
  to'liqlik farqi B - A = +0.07 [0.04; 0.10],  farq <= 0 bo'lgan holatlar: 0.0%



## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Sinov to'plamida 30 ta hujum:
  to'liqlik  A = 0.67 [0.48; 0.83]   B = 0.77 [0.59; 0.91]
  aniqlik    A = 0.34 [0.22; 0.47]   B = 0.31 [0.21; 0.42]
  to'liqlik farqi B - A = +0.10 [0.00; 0.21],  farq <= 0 bo'lgan holatlar: 4.9%

Sinov to'plamida 300 ta hujum:
  to'liqlik  A = 0.82 [0.77; 0.86]   B = 0.89 [0.85; 0.92]
  aniqlik    A = 0.85 [0.81; 0.89]   B = 0.79 [0.75; 0.84]
  to'liqlik farqi B - A = +0.07 [0.04; 0.10],  farq <= 0 bo'lgan holatlar: 0.0%
```

## Natijalar tahlili


1. *Kichik sinov to'plami aldaydi.* 30 ta hujumli to'plamda A modelning o'lchangan to'liqligi 0,67 chiqdi, holbuki uning haqiqiy qiymati taxminan 0,80. Ishonch oralig'i [0,48; 0,83] ga cho'zilgan, ya'ni bu to'plam asosida modelning sifati haqida deyarli hech narsa deb bo'lmaydi.
2. *Modellarni taqqoslash.* 30 ta hujumda B modelning ustunligi chegarada: farqning ishonch oralig'i noldan boshlanadi va bootstrap tanlovlarining qariyb 5 foizida farq yo'q. 300 ta hujumda esa farq [0,04; 0,10] oralig'ida va ishonchli.
3. *Aniqlik bazaviy chastotaga bog'liq.* 30 ta hujumli to'plamda ikkala modelning aniqligi ham 0,35 dan past, 300 ta hujumlida esa 0,8 dan yuqori, holbuki modellarning o'zi o'zgarmagan. Sababi — oddiy yozuvlar soni ikkala holatda ham 10 000 ta, demak noto'g'ri signallar soni bir xil, haqiqiy hujumlar esa o'n barobar kam. Bu hodisa 5.3-bo'limda bazaviy chastota xatosi sifatida batafsil ko'rib chiqiladi.

Amaliy xulosa: modellar natijalarini taqqoslashda har doim sinov to'plamidagi hujumlar sonini va ishonch oraliqlarini ko'rsating. Sinov to'plamida bir necha o'nta hujum bo'lsa, 2–3 foiz punktlik farqlar asosida xulosa chiqarib bo'lmaydi.

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 5-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.